In [1]:
using Pkg
Pkg.activate("../../")
Pkg.instantiate()

im_path = "../../../Images/64/"

using Lux, LuxCUDA, MLUtils
using Optimisers, Random, Statistics
using Zygote
using DiffEqFlux, OrdinaryDiffEq
using FFTW
#using Reactant
using Images, JLD2
using ComponentArrays
using DeepEquilibriumNetworks
using Plots
using Dates

  Activating project at `~/Code/ImplicitDenoiser`


In [2]:
#Reactant.set_default_backend("cpu")
#const device = reactant_device()
const cdev = cpu_device()
const gdev = gpu_device()
dev = gdev


(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

In [3]:
include("model.jl")


Chain(
    layer_1 = Chain(
        layer_1 = Conv((5, 5), 1 => 8, pad=2),    # 208 parameters
        layer_2 = Conv((5, 5), 8 => 16, pad=2),   # 3_216 parameters
        layer_3 = SkipConnection(
            connection = +,
            layers = Chain(
                layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
            ),
        ),
    ),
    layer_2 = GroupNorm(16, 4, affine=true),      # 32 parameters
    layer_3 = Conv((5, 5), 16 => 32, pad=2),      # 12_832 parameters
    layer_4 = Conv((5, 5), 32 => 64, pad=2),      # 51_264 parameters
    layer_5 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 64 => 128, gelu_tanh),  # 8_320 parameters
            layer_2 = Conv((1, 1), 128 => 128, gelu_tanh),  # 16_512 parameters
            layer_3 = Conv((1, 1), 128 =>

In [4]:
@load "ps_latest.jld2" ps
@load "st_latest.jld2" st

1-element Vector{Symbol}:
 :st

In [5]:
ps = ps |> ComponentArray |> dev
st = st |> dev

(layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer_2 = NamedTuple(), layer_3 = NamedTuple(), layer_4 = NamedTuple(), layer_5 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple()), layer_6 = NamedTuple(), layer_7 = NamedTuple(), layer_8 = NamedTuple(), layer_9 = NamedTuple())

In [6]:
model = NeuralODE(model_conv, (0.0f0, 1.0f0), Tsit5(); save_everystep = false, sensealg = BacksolveAdjoint(; autojacvec = ZygoteVJP()), reltol = 1e-3, abstol = 1e-4, save_start = false)

NeuralODE(
    model = Chain(
        layer_1 = Chain(
            layer_1 = Conv((5, 5), 1 => 8, pad=2),  # 208 parameters
            layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
            layer_3 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                    layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                    layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                ),
            ),
        ),
        layer_2 = GroupNorm(16, 4, affine=true),  # 32 parameters
        layer_3 = Conv((5, 5), 16 => 32, pad=2),  # 12_832 parameters
        layer_4 = Conv((5, 5), 32 => 64, pad=2),  # 51_264 parameters
        layer_5 = SkipConnection(
            connection = +,
            layers = Chain(
                layer_1 = Conv((1, 1), 64 => 128, gelu_tanh),  # 8_320 parameters
                layer_2 = Conv((1, 1)

In [7]:
opt = Optimisers.NAdam(1e-4)
state = Optimisers.setup(opt,ps)
train_state = Lux.Training.TrainState(model, ps, st, opt)

TrainState(
    NeuralODE(
        model = Chain(
            layer_1 = Chain(
                layer_1 = Conv((5, 5), 1 => 8, pad=2),  # 208 parameters
                layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                layer_3 = SkipConnection(
                    connection = +,
                    layers = Chain(
                        layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                        layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                        layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                    ),
                ),
            ),
            layer_2 = GroupNorm(16, 4, affine=true),  # 32 parameters
            layer_3 = Conv((5, 5), 16 => 32, pad=2),  # 12_832 parameters
            layer_4 = Conv((5, 5), 32 => 64, pad=2),  # 51_264 parameters
            layer_5 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_1

In [8]:
function loss_function(model, ps, st, (x, y_true); tspan = (0.0f0, 1.0f0))
    y, st = model(x, ps, st; tspan = tspan)
    y_pred = y
    #y_pred = model(x, ps, st)[1][1]
    loss_mse= MSELoss()
    mse_loss = loss_mse(y_pred, y_true)
    #sptrl_loss = loss_mse(dct(dct(y_pred,1),2),dct(dct(y_pred,1),2))
    return mse_loss, st
    #return mes_loss + sptrl_loss, st
end

loss_function (generic function with 1 method)

In [9]:
x = randn(Float32,128,128,1,4)
y_true = randn(Float32,128,128,1,4)
x_dev = x |> dev
y_dev = y_true |> dev
y, _ = model(x_dev, ps, st; tspan = (0.0f0, 1.0f0))

MethodError: MethodError: no method matching (::NeuralODE{Chain{@NamedTuple{layer_1::Chain{@NamedTuple{layer_1::Conv{typeof(identity), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_2::Conv{typeof(identity), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_3::SkipConnection{Chain{@NamedTuple{layer_1::Conv{typeof(NNlib.gelu_tanh), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_2::Conv{typeof(NNlib.gelu_tanh), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_3::Conv{typeof(identity), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, In

In [12]:
fieldnames(typeof(model.tspan))

(1, 2)

In [20]:
model.tspan = (0.0f0, 2.0f0)

ErrorException: setfield!: immutable struct of type NeuralODE cannot be changed

In [86]:
randn()

-0.3730985395635301